# DynGraphEval

Evaluates temporal graph models on two dimensions:
1. **Standard MRR** — TGB `hist_rnd` negatives (leaderboard-comparable)
2. **Recency MRR** — negatives from the source node's recent interaction history

**To switch models:** edit `config.yaml`, then re-run all cells.

## 0. Colab Setup
Run this cell first. It mounts Google Drive, installs dependencies, and makes `dyngrapheval` importable. Skip if running locally.

In [ ]:
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # ── Mount Google Drive ────────────────────────────────────────────────────
    # Your DynGraphEval folder must be in Drive at the path below.
    # Adjust REPO_PATH to wherever you placed it.
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_PATH = "/content/drive/MyDrive/DynGraphEval"  # ← edit if needed
    os.chdir(REPO_PATH)
    print(f"Working directory: {os.getcwd()}")

    # ── Install dependencies ──────────────────────────────────────────────────
    # torch-scatter requires a wheel matching your CUDA version.
    # The URL below targets CUDA 12.1 (default Colab GPU as of 2024).
    # Check your version with: !nvcc --version
    import torch
    cu = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
    print(f"Installing for: torch {torch.__version__}, {cu}")

    os.system(f"pip install -q torch-geometric py-tgb pyyaml tqdm")
    os.system(
        f"pip install -q torch-scatter "
        f"-f https://data.pyg.org/whl/torch-{torch.__version__}+{cu}.html"
    )

    # ── Make dyngrapheval importable ──────────────────────────────────────────
    # Adds src/ to the Python path so `import dyngrapheval` works
    # without needing to pip install -e .
    src_path = os.path.join(REPO_PATH, "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    print("Setup complete.")
else:
    print("Running locally — skipping Colab setup.")

## 1. Imports

In [ ]:
import json
import yaml
import torch
import numpy as np

from tgb.linkproppred.dataset_pyg import PyGLinkPropPredDataset

from dyngrapheval import Evaluator
from dyngrapheval.models import TGN, FederatedTGN, FedLink

## 2. Load Config

In [ ]:
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

# Read all settings (with defaults for optional fields)
model_type    = cfg["model"]                        # tgn | fl_tgn | fedlink
checkpoints   = cfg["checkpoints"]                  # str or list[str]
model_name    = cfg.get("name",      model_type)
dataset_name  = cfg.get("dataset",   "tgbl-wiki")
seed          = cfg.get("seed",      42)
neg_cache_dir = cfg.get("cache_dir", "neg_cache")
device_str    = cfg.get("device",    None)          # None = auto-detect

# Normalize checkpoints to always be a list
if isinstance(checkpoints, str):
    checkpoints = [checkpoints]

print(f"Model:    {model_name} ({model_type})")
print(f"Dataset:  {dataset_name}")
print(f"Checkpoints: {checkpoints}")

## 3. Device & Dataset

In [ ]:
# ── Device ────────────────────────────────────────────────────────────────────
if device_str:
    device = torch.device(device_str)
elif torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

# ── Dataset ───────────────────────────────────────────────────────────────────
dataset = PyGLinkPropPredDataset(name=dataset_name, root="datasets")
data    = dataset.get_TemporalData()

# Fix for pandas 3.x: Arrow-backed arrays are read-only, TGB needs writable copies
dataset._data.src = np.asarray(dataset._data.src, dtype=float)
dataset._data.dst = np.asarray(dataset._data.dst, dtype=float)
dataset._data.ts  = np.asarray(dataset._data.ts,  dtype=float)

# Temporal splits
split_mask = dataset.get_idx_split()
train_data = data[split_mask["train"]]
val_data   = data[split_mask["valid"]]
test_data  = data[split_mask["test"]]

# Metadata
num_nodes   = dataset.num_nodes
msg_dim     = dataset.edge_feat_dim
min_dst_idx = int(data.dst.min().item())
max_dst_idx = int(data.dst.max().item())
num_users   = int(data.src.max().item()) + 1
num_pages   = max_dst_idx - min_dst_idx + 1

print(f"Nodes: {num_nodes}  |  Edges: {data.num_events}")
print(f"Train: {train_data.num_events}  Val: {val_data.num_events}  Test: {test_data.num_events}")

## 4. Instantiate Model

In [ ]:
if model_type == "tgn":
    # Centralized TGN — single model, single checkpoint.
    # all_t / all_msg are the full dataset's edge tensors.
    # The neighbor_loader returns e_ids that index into these tensors
    # to retrieve timestamps and features for the GNN attention layer.
    model = TGN(
        checkpoint_path = checkpoints[0],
        num_nodes       = num_nodes,
        msg_dim         = msg_dim,
        all_t           = data.t,
        all_msg         = data.msg,
        device          = device,
    )

elif model_type == "fl_tgn":
    # Federated TGN — 4 independent clients, one checkpoint each.
    # Data is partitioned by source node ID (same split as training).
    # Global MRR = weighted average across clients by edge count.
    model = FederatedTGN(
        checkpoint_paths = checkpoints,
        num_nodes        = num_nodes,
        msg_dim          = msg_dim,
        train_data       = train_data,
        val_data         = val_data,
        test_data        = test_data,
        device           = device,
    )

elif model_type == "fedlink":
    # FedLink — static GraphSAGE, no temporal reasoning.
    # warmup() is a no-op; embeddings are recomputed fresh each eval call.
    model = FedLink(
        checkpoint_paths = checkpoints,
        num_users        = num_users,
        num_pages        = num_pages,
        train_data       = train_data,
        val_data         = val_data,
        test_data        = test_data,
        min_dst_idx      = min_dst_idx,
        device           = device,
    )

else:
    raise ValueError(f"Unknown model type: '{model_type}'. Choose from: tgn, fl_tgn, fedlink")

model.load_checkpoint()
print(f"Loaded: {model_name}")

## 5. Run Evaluation

In [ ]:
evaluator = Evaluator(
    dataset       = dataset,
    train_data    = train_data,
    val_data      = val_data,
    test_data     = test_data,
    first_dst_id  = min_dst_idx,
    last_dst_id   = max_dst_idx,
    dataset_name  = dataset_name,
    neg_cache_dir = neg_cache_dir,
    seed          = seed,
)

results = evaluator.run(model, model_name=model_name)

## 6. Results

In [ ]:
print(json.dumps(results, indent=2))